In [ ]:
import json
import sqlite3

import mlflow
from openai import OpenAI

In [ ]:
# Specify the tracking URI for the MLflow server.
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [ ]:
mlflow.set_experiment("music_agent")
mlflow.openai.autolog()

client = OpenAI()

In [ ]:
# our agent has one tool: run SQL on the music database
db = sqlite3.connect("file:../data/music.db?mode=ro", uri=True)


@mlflow.trace(span_type="TOOL")
def run_sql(query: str) -> str:
    try:
        return str(db.execute(query).fetchall())
    except Exception as e:
        return f"Error: {e}"


tools = [
    {
        "type": "function",
        "name": "run_sql",
        "description": "Run a SQL query on the music database (SQLite)",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    }
]

In [ ]:
# the agent: call the model, run the tools it asks for, repeat until it answers
# @mlflow.trace groups all the LLM calls and tool calls of one question into a single trace
@mlflow.trace(span_type="AGENT")
def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]

    for _ in range(10):
        response = client.responses.create(
            model="gpt-6-luna",
            instructions="You answer questions about a music store. Look at the tables and their columns before writing a query.",
            input=messages,
            tools=tools,
        )
        messages += response.output

        tool_calls = [item for item in response.output if item.type == "function_call"]
        if not tool_calls:
            return response.output_text

        for call in tool_calls:
            result = run_sql(**json.loads(call.arguments))
            messages.append({"type": "function_call_output", "call_id": call.call_id, "output": result})

In [ ]:
ask("Which genre on average has the longest tracks?")

In [ ]:
ask("List the 5 artists with the most albums.")

In [ ]:
ask("Who is the best-selling artist, and what is their best-selling album?")